# Neutron emission probability demo

This notebook exercises `calculate_neutron_emission_probabilities` with both average values and radial profiles.

In [21]:
from pathlib import Path
import sys
import importlib

# Add repo src/ to sys.path so package imports work in a notebook
nb_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
src_root = nb_dir.parent.parent  # .../src
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

import fusion_fuel_concepts_explorer.sources.neutron_source as neutron_source
importlib.reload(neutron_source)  # ensure latest version when re-running cells
calculate_neutron_emission_probabilities = neutron_source.calculate_neutron_emission_probabilities

## Case 1: Average values (fractions)
- D/T mix, 15 keV, uniform density and volume
- Fractions are normalized automatically

In [22]:

res_avg = calculate_neutron_emission_probabilities(
    {"D": 0.5, "He3": 0.5},
    ion_temperature_keV=70.0,
    total_ion_density_cm3=1e14,
    volume_profile_cm3=1.5e6,
    reactivity_model="bosch_hale",
    include_secondary_equilibrium=True
)
res_avg.probabilities, res_avg.total_neutron_rate_s

({'2.45 MeV (DDn)': 0.5364090783848406,
  '14.1 MeV (DTn)': 0.4571007395919094,
  '~11 MeV (TTn)': 2.7325449698270928e-05,
  '~12 MeV (THe3 n)': 0.006462856573551832},
 6.158101132490711e+16)

In [23]:

res_avg = calculate_neutron_emission_probabilities(
    {"D": 0.5, "T": 0.5},
    ion_temperature_keV=14.0,
    total_ion_density_cm3=1e14,
    volume_profile_cm3=1.5e6,
    reactivity_model="bosch_hale",
    include_secondary_equilibrium=True
)
res_avg.probabilities, res_avg.total_neutron_rate_s

({'2.45 MeV (DDn)': 0.002644608269428239,
  '14.1 MeV (DTn)': 0.9944663889207973,
  '~11 MeV (TTn)': 0.0027303138161938077,
  '~12 MeV (THe3 n)': 0.00015868899358063393},
 9.093385583350043e+17)

## Case 2: Radial profiles (absolute densities)
- Three radial zones with shell volumes
- Absolute densities provided; total density is not needed

In [24]:
res_profile = calculate_neutron_emission_probabilities(
    {"D": [3e13, 2e13, 1e13], "T": [2e13, 1.5e13, 1e13]},
    ion_temperature_keV=[8, 10, 12],
    total_ion_density_cm3=None,
    volume_profile_cm3=[5e4, 5e4, 5e4],
    absolute_densities=True,
    reactivity_model="bosch_hale",
)
res_profile.probabilities, res_profile.reaction_rates_s, res_profile.total_neutron_rate_s

({'2.45 MeV (DDn)': 0.0035869243740994385,
  '14.1 MeV (DTn)': 0.993853599308662,
  '~11 MeV (TTn)': 0.0024227192739346183,
  '~12 MeV (THe3 n)': 0.00013675704330389356},
 {'DDp_rate_s': 15507326496964.043,
  'DDn_2.45MeV': 16095815829956.344,
  'DTn_14.1MeV': 4459777466155180.5,
  'DHe3_rate_s': 16095815829956.344,
  'TT_rate_s': 10871610096526.7,
  'THe3_rate_s': 613677890273.7501},
 4487358569971937.5)

Quick check: probabilities sum to 1.

In [25]:
sum(res_avg.probabilities.values()), sum(res_profile.probabilities.values())

(1.0, 1.0)